# S3 Data Intelligence with Snowflake Cortex AI

**Build an AI-powered document intelligence pipeline in 60 minutes.**

```
S3 (PDFs) → AI Functions → Intelligence Table → Cortex Search + Analyst → Agent → Chat
```

**What you'll do:**
1. Connect Snowflake to your S3 bucket
2. Parse and enrich PDFs with 7 Cortex AI functions
3. Enable semantic search over documents
4. Query structured data with natural language
5. Chat with a unified agent that combines both

**Prerequisites:**
- S3 bucket with `healthcare/pdfs/` prefix containing sample PDFs
- IAM role for Snowflake (see LAB_GUIDE.md)

---

## Step 1: Setup (2 min)

In [ ]:
%%sql -r setup_result
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE HEALTHCARE_AI_DEMO;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.RAW;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.PROCESSED;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.ANALYTICS;

CREATE OR REPLACE WAREHOUSE HEALTHCARE_AI_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE HEALTHCARE_AI_WH;
USE DATABASE HEALTHCARE_AI_DEMO;

---
## Step 2: Connect to S3 (5 min)

**Replace** the two placeholders below with your values, then run.

> Do NOT re-run this cell after updating the trust policy — it generates a new external ID each time.

In [ ]:
%%sql -r integration_result
CREATE STORAGE INTEGRATION IF NOT EXISTS HEALTHCARE_S3_INTEGRATION
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = '<YOUR_AWS_IAM_ROLE_ARN>'
  STORAGE_ALLOWED_LOCATIONS = ('s3://<YOUR_BUCKET_NAME>/healthcare/');

### Update IAM Trust Policy

From the output below, copy **`STORAGE_AWS_IAM_USER_ARN`** and **`STORAGE_AWS_EXTERNAL_ID`**.

Go to **AWS Console → IAM → Roles → your role → Trust relationships → Edit trust policy** and set:

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": { "AWS": "<STORAGE_AWS_IAM_USER_ARN>" },
    "Action": "sts:AssumeRole",
    "Condition": { "StringEquals": { "sts:ExternalId": "<STORAGE_AWS_EXTERNAL_ID>" } }
  }]
}
```

**Save**, wait 30 seconds, then continue.

In [ ]:
%%sql -r describe_result
DESCRIBE INTEGRATION HEALTHCARE_S3_INTEGRATION;

In [ ]:
%%sql -r stage_result
CREATE OR REPLACE STAGE RAW.S3_MEDICAL_DOCS
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/pdfs/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

### Verify — should list your PDF files

In [ ]:
%%sql -r list_result
LIST @RAW.S3_MEDICAL_DOCS;

---
## Step 3: Explore AI Functions (15 min)

Let's see each Cortex AI function in action on a real PDF before building the full pipeline.

### 3a. AI_PARSE_DOCUMENT — Extract text from PDF

In [ ]:
%%sql -r parse_result
SELECT
  RELATIVE_PATH AS FILE_NAME,
  AI_PARSE_DOCUMENT(
    TO_FILE('@RAW.S3_MEDICAL_DOCS', RELATIVE_PATH),
    OBJECT_CONSTRUCT('mode', 'OCR')
  ):content::VARCHAR AS PARSED_TEXT
FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS)
LIMIT 1;

### 3b. AI_EXTRACT — Pull structured fields from free text

In [ ]:
%%sql -r extract_result
WITH parsed AS (
  SELECT RELATIVE_PATH AS FILE_NAME,
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR AS txt
  FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS) LIMIT 1
)
SELECT FILE_NAME,
  AI_EXTRACT(txt, OBJECT_CONSTRUCT(
    'patient_name', 'string: full name',
    'diagnosis', 'string: primary diagnosis',
    'medications', 'array: list of medications',
    'provider_name', 'string: doctor name'
  )) AS EXTRACTED
FROM parsed;

### 3c. AI_CLASSIFY — Categorize documents (zero-shot)

In [ ]:
%%sql -r classify_result
SELECT
  RELATIVE_PATH AS FILE_NAME,
  AI_CLASSIFY(
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR,
    ARRAY_CONSTRUCT('Lab Report','Discharge Summary','Prescription','Radiology Report','Clinical Notes')
  ):labels[0]::VARCHAR AS CATEGORY
FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS)
LIMIT 3;

### 3d. SUMMARIZE — Concise summary

In [ ]:
%%sql -r summary_result
SELECT
  RELATIVE_PATH AS FILE_NAME,
  SNOWFLAKE.CORTEX.SUMMARIZE(
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR
  ) AS SUMMARY
FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS)
LIMIT 1;

### 3e. AI_REDACT — Remove PII

In [ ]:
%%sql -r redact_result
SELECT
  RELATIVE_PATH AS FILE_NAME,
  AI_REDACT(
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR
  ) AS REDACTED_TEXT
FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS)
LIMIT 1;

---
## Step 4: Build the Intelligence Table (10 min)

One query processes all PDFs through 7 AI functions and stores the results. No pipes, no procedures — just a single `INSERT ... SELECT` from `DIRECTORY()`.

In [ ]:
%%sql -r create_table_result
CREATE OR REPLACE TABLE PROCESSED.PDF_INTELLIGENCE (
    DOC_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PARSED_TEXT VARCHAR(16777216),
    EXTRACTED_FIELDS VARIANT,
    DOC_CATEGORY VARCHAR(100),
    SENTIMENT_SCORE FLOAT,
    SUMMARY VARCHAR(10000),
    REDACTED_TEXT VARCHAR(16777216),
    KEY_INSIGHTS VARCHAR(10000),
    EMBEDDING VECTOR(FLOAT, 768)
);

### Process all PDFs (takes ~8 min for 6 files)

This single statement reads every PDF from S3 and applies all AI functions:

In [ ]:
%%sql -r process_result
INSERT INTO PROCESSED.PDF_INTELLIGENCE (
    FILE_NAME, PARSED_TEXT, EXTRACTED_FIELDS, DOC_CATEGORY,
    SENTIMENT_SCORE, SUMMARY, REDACTED_TEXT, KEY_INSIGHTS, EMBEDDING)
SELECT
    d.RELATIVE_PATH,
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR,
    AI_EXTRACT(
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR,
      OBJECT_CONSTRUCT(
        'patient_name','string: full name','diagnosis','string: primary diagnosis',
        'medications','array: medications','provider_name','string: doctor name',
        'date_of_service','string: date','follow_up','string: follow-up instructions'
      )
    ),
    AI_CLASSIFY(
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR,
      ARRAY_CONSTRUCT('Lab Report','Discharge Summary','Prescription','Radiology Report','Clinical Notes','Pathology Report')
    ):labels[0]::VARCHAR,
    SNOWFLAKE.CORTEX.SENTIMENT(
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR
    ),
    SNOWFLAKE.CORTEX.SUMMARIZE(
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR
    ),
    AI_REDACT(
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR
    ),
    AI_COMPLETE('mistral-large2', CONCAT(
      'Provide 3 key clinical insights from this document:\n',
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR
    )),
    AI_EMBED('snowflake-arctic-embed-m-v1.5',
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', d.RELATIVE_PATH), OBJECT_CONSTRUCT('mode','OCR')):content::VARCHAR
    )
FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS) d
WHERE d.RELATIVE_PATH != '';

### Explore the results

In [ ]:
%%sql -r explore_result
SELECT FILE_NAME, DOC_CATEGORY, SENTIMENT_SCORE,
  EXTRACTED_FIELDS:patient_name::VARCHAR AS PATIENT,
  EXTRACTED_FIELDS:diagnosis::VARCHAR AS DIAGNOSIS,
  LEFT(SUMMARY, 200) AS SUMMARY_PREVIEW
FROM PROCESSED.PDF_INTELLIGENCE;

---
## Step 5: Cortex Search — Semantic Retrieval (3 min)

Cortex Search indexes your enriched text so you can search by *meaning*. This powers the agent's document retrieval tool.

In [ ]:
%%sql -r search_result
CREATE OR REPLACE CORTEX SEARCH SERVICE PROCESSED.PDF_SEARCH
  ON SEARCH_TEXT
  ATTRIBUTES DOC_CATEGORY, PATIENT_NAME, DIAGNOSIS
  WAREHOUSE = HEALTHCARE_AI_WH
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-m-v1.5'
AS (
    SELECT DOC_ID, FILE_NAME,
      CONCAT(COALESCE(PARSED_TEXT,''), '\nSUMMARY: ', COALESCE(SUMMARY,''), '\nINSIGHTS: ', COALESCE(KEY_INSIGHTS,'')) AS SEARCH_TEXT,
      DOC_CATEGORY,
      COALESCE(EXTRACTED_FIELDS:patient_name::VARCHAR, 'Unknown') AS PATIENT_NAME,
      COALESCE(EXTRACTED_FIELDS:diagnosis::VARCHAR, '') AS DIAGNOSIS
    FROM PROCESSED.PDF_INTELLIGENCE
);

---
## Step 6: Structured Data + Semantic View (5 min)

Load sample healthcare data and create a Semantic View so Cortex Analyst can query it with natural language.

In [ ]:
%%sql -r data_result
CREATE OR REPLACE TABLE ANALYTICS.PROVIDERS (
    PROVIDER_ID NUMBER PRIMARY KEY, PROVIDER_NAME VARCHAR, SPECIALTY VARCHAR(100),
    FACILITY_NAME VARCHAR(200), CITY VARCHAR(100), STATE VARCHAR(50)
);
INSERT INTO ANALYTICS.PROVIDERS VALUES
  (1,'Dr. Sarah Chen','Internal Medicine','Mercy General Hospital','Hartford','CT'),
  (2,'Dr. James Okafor','Cardiology','Mercy General Hospital','Hartford','CT'),
  (3,'Dr. Maria Rodriguez','Orthopedics','St. Luke''s Medical Center','New Haven','CT'),
  (4,'Dr. Robert Kim','Neurology','Yale New Haven Hospital','New Haven','CT'),
  (5,'Dr. Aisha Patel','Oncology','Hartford Healthcare Cancer','Hartford','CT'),
  (6,'Dr. David Thompson','Psychiatry','Silver Hill Hospital','New Canaan','CT');

CREATE OR REPLACE TABLE ANALYTICS.PATIENTS (
    PATIENT_ID NUMBER PRIMARY KEY, FIRST_NAME VARCHAR(100), LAST_NAME VARCHAR(100),
    INSURANCE_PLAN VARCHAR(200), PRIMARY_PROVIDER_ID NUMBER
);
INSERT INTO ANALYTICS.PATIENTS VALUES
  (101,'John','Whitfield','Aetna PPO',1),(102,'Margaret','Sullivan','UnitedHealth',2),
  (103,'David','Park','Cigna',3),(104,'Sandra','Williams','Anthem Blue Cross',1),
  (105,'Roberto','Garcia','Medicare Advantage',2),(106,'Linda','Brown','Medicare Advantage',5);

CREATE OR REPLACE TABLE ANALYTICS.CLAIMS (
    CLAIM_ID NUMBER PRIMARY KEY, PATIENT_ID NUMBER, PROVIDER_ID NUMBER,
    SERVICE_DATE DATE, PROCEDURE_DESC VARCHAR(500), DIAGNOSIS_DESC VARCHAR(500),
    BILLED_AMOUNT NUMBER(10,2), PAID_AMOUNT NUMBER(10,2), CLAIM_STATUS VARCHAR(50)
);
INSERT INTO ANALYTICS.CLAIMS VALUES
  (1001,101,1,'2024-06-10','Office visit','Essential hypertension',185.00,113.60,'Approved'),
  (1002,101,2,'2024-06-15','Electrocardiogram','Essential hypertension',350.00,224.00,'Approved'),
  (1003,102,2,'2024-07-01','Echocardiography','Atherosclerotic heart disease',1200.00,768.00,'Approved'),
  (1004,103,3,'2024-07-05','Knee arthroscopy','Torn meniscus',4500.00,2880.00,'Approved'),
  (1005,104,1,'2024-07-10','Office visit','Type 2 diabetes',150.00,92.00,'Approved'),
  (1006,102,2,'2024-08-01','Cardiac catheterization','Atherosclerotic heart disease',45000.00,28800.00,'Approved'),
  (1007,105,2,'2024-08-15','Stress test','Chest pain',2500.00,1600.00,'Approved'),
  (1008,103,3,'2024-09-01','Physical therapy','Torn meniscus',350.00,224.00,'Denied'),
  (1009,106,5,'2024-09-10','Chemotherapy','Breast cancer',425.00,272.00,'Approved'),
  (1010,101,1,'2024-09-20','Follow-up visit','Essential hypertension',150.00,96.00,'Approved');

In [ ]:
%%sql -r semantic_result
CREATE OR REPLACE SEMANTIC VIEW ANALYTICS.HEALTHCARE_ANALYTICS
  TABLES (
    PATIENTS AS ANALYTICS.PATIENTS PRIMARY KEY (PATIENT_ID) COMMENT = 'Patient demographics',
    PROVIDERS AS ANALYTICS.PROVIDERS PRIMARY KEY (PROVIDER_ID) COMMENT = 'Doctor directory',
    CLAIMS AS ANALYTICS.CLAIMS PRIMARY KEY (CLAIM_ID) COMMENT = 'Medical claims and billing'
  )
  RELATIONSHIPS (
    claims_to_patients AS CLAIMS (PATIENT_ID) REFERENCES PATIENTS,
    claims_to_providers AS CLAIMS (PROVIDER_ID) REFERENCES PROVIDERS
  )
  DIMENSIONS (
    PATIENTS.first_name AS FIRST_NAME COMMENT = 'Patient first name',
    PATIENTS.last_name AS LAST_NAME COMMENT = 'Patient last name',
    PATIENTS.insurance_plan AS INSURANCE_PLAN COMMENT = 'Insurance plan',
    PROVIDERS.provider_name AS PROVIDER_NAME COMMENT = 'Doctor name',
    PROVIDERS.specialty AS SPECIALTY COMMENT = 'Medical specialty',
    CLAIMS.procedure_desc AS PROCEDURE_DESC COMMENT = 'Procedure performed',
    CLAIMS.diagnosis_desc AS DIAGNOSIS_DESC COMMENT = 'Diagnosis',
    CLAIMS.claim_status AS CLAIM_STATUS COMMENT = 'Approved or Denied',
    CLAIMS.service_date AS SERVICE_DATE COMMENT = 'Date of service'
  )
  METRICS (
    CLAIMS.total_billed AS SUM(BILLED_AMOUNT) COMMENT = 'Total billed in dollars',
    CLAIMS.total_paid AS SUM(PAID_AMOUNT) COMMENT = 'Total paid by insurance',
    CLAIMS.claim_count AS COUNT(CLAIM_ID) COMMENT = 'Number of claims',
    CLAIMS.avg_billed AS AVG(BILLED_AMOUNT) COMMENT = 'Average billed amount'
  )
  COMMENT = 'Healthcare analytics for Cortex Analyst';

---
## Step 7: Create the Cortex Agent (3 min)

The agent combines **Cortex Analyst** (structured SQL queries) with **Cortex Search** (document retrieval) into one conversational interface.

In [ ]:
%%sql -r agent_result
CREATE OR REPLACE AGENT ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT
  FROM SPECIFICATION
$$
models:
  orchestration: auto
instructions:
  system: >
    You are a healthcare intelligence assistant. You help users explore patient data,
    claims, and medical documents. Be concise and clinically relevant.
  orchestration: >
    Use HealthcareAnalyst for quantitative questions about patients, providers, claims, and costs.
    Use PDFSearch for questions about medical document content, diagnoses, and clinical findings.
  sample_questions:
    - question: "Which providers have the highest total billed amounts?"
    - question: "Find documents mentioning diabetes"
    - question: "What is the average claim amount by specialty?"
    - question: "Show denied claims and their reasons"
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: HealthcareAnalyst
      description: "Queries structured healthcare data: patients, providers, claims, costs."
  - tool_spec:
      type: cortex_search
      name: PDFSearch
      description: "Searches AI-processed PDF medical documents for clinical content."
tool_resources:
  HealthcareAnalyst:
    semantic_view: "HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_ANALYTICS"
    execution_environment:
      type: warehouse
      warehouse: "HEALTHCARE_AI_WH"
  PDFSearch:
    name: "HEALTHCARE_AI_DEMO.PROCESSED.PDF_SEARCH"
$$;

---
## Step 8: Test the Agent (10 min)

### Option A: Snowflake Intelligence (recommended)
Go to **AI & ML → Intelligence** → select `HEALTHCARE_INTELLIGENCE_AGENT` → start chatting.

### Option B: SQL

**Structured query** (routes to Cortex Analyst → generates SQL):

In [ ]:
%%sql -r agent_test_1
SELECT TRY_PARSE_JSON(
  SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
    $${"messages": [{"role": "user", "content": [{"type": "text", "text": "Which providers have the highest total billed amounts?"}]}]}$$
  )
) AS RESPONSE;

**Document search** (routes to Cortex Search):

In [ ]:
%%sql -r agent_test_2
SELECT TRY_PARSE_JSON(
  SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
    $${"messages": [{"role": "user", "content": [{"type": "text", "text": "Find documents mentioning hypertension"}]}]}$$
  )
) AS RESPONSE;

---
## Step 9: Deploy Streamlit Chat App (5 min)

Create a shareable chat interface powered by the agent.

In [ ]:
%%sql -r streamlit_stage
CREATE OR REPLACE STAGE ANALYTICS.STREAMLIT_APP
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
from snowflake.snowpark.context import get_active_session
import io

session = get_active_session()

app_code = """import streamlit as st
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()
st.set_page_config(page_title="Healthcare Intelligence", page_icon="\U0001f3e5")
st.title("\U0001f3e5 Healthcare Intelligence Agent")
st.caption("Ask questions about patients, claims, or medical documents")

AGENT = "HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT"

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Ask anything..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    payload = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}]})
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            sql = "SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('" + AGENT + "', $$" + payload + "$$)"
            result = session.sql(sql).collect()
            response = json.loads(result[0][0])
            parts = [i["text"] for i in response.get("content", []) if i.get("type") == "text"]
            answer = chr(10).join(parts) if parts else "No response."
            st.markdown(answer)
            st.session_state.messages.append({"role": "assistant", "content": answer})
"""

env_yml = "name: streamlit\nchannels:\n  - snowflake\ndependencies:\n  - streamlit\n  - snowflake-snowpark-python\n"

session.file.put_stream(
    io.BytesIO(app_code.encode()), "@HEALTHCARE_AI_DEMO.ANALYTICS.STREAMLIT_APP/streamlit_app.py",
    auto_compress=False, overwrite=True)
session.file.put_stream(
    io.BytesIO(env_yml.encode()), "@HEALTHCARE_AI_DEMO.ANALYTICS.STREAMLIT_APP/environment.yml",
    auto_compress=False, overwrite=True)
print("Streamlit files uploaded.")

In [ ]:
%%sql -r streamlit_app
CREATE OR REPLACE STREAMLIT ANALYTICS.HEALTHCARE_CHAT_APP
  ROOT_LOCATION = '@HEALTHCARE_AI_DEMO.ANALYTICS.STREAMLIT_APP'
  MAIN_FILE = 'streamlit_app.py'
  QUERY_WAREHOUSE = 'HEALTHCARE_AI_WH'
  TITLE = 'Healthcare Intelligence Chat';

### Done!

You now have:
- **Cortex Agent** → AI & ML → Intelligence → select the agent
- **Streamlit App** → Projects → Streamlit → Healthcare Intelligence Chat

Both are shareable with anyone in your Snowflake account.

---
## Cleanup

In [ ]:
%%sql -r cleanup_result
-- Uncomment to remove all lab objects:
-- DROP DATABASE IF EXISTS HEALTHCARE_AI_DEMO;
-- DROP WAREHOUSE IF EXISTS HEALTHCARE_AI_WH;
-- DROP STORAGE INTEGRATION IF EXISTS HEALTHCARE_S3_INTEGRATION;